<a href="https://colab.research.google.com/github/AndreiMoraru123/learning_ray/blob/main/notebooks/ch_08_model_serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Online Inference with Ray Serve


You can run this notebook directly in
[Colab](https://colab.research.google.com/github/maxpumperla/learning_ray/blob/main/notebooks/ch_08_model_serving.ipynb).
<a target="_blank" href="https://colab.research.google.com/github/maxpumperla/learning_ray/blob/main/notebooks/ch_08_model_serving.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For this chapter you need to install the following dependencies:

In [1]:
! pip install "ray[serve]" "transformers"
! pip install "requests" "wikipedia"


To import utility files for this chapter, on Colab you will also have to clone
the repo and copy the code files to the base path of the runtime:

In [2]:
!git clone https://github.com/maxpumperla/learning_ray
%cp -r learning_ray/notebooks/* .

Cloning into 'learning_ray'...
remote: Enumerating objects: 1385, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 1385 (delta 9), reused 5 (delta 5), pack-reused 1371 (from 2)
Receiving objects: 100% (1385/1385), 119.79 MiB | 39.38 MiB/s, done.
Resolving deltas: 100% (753/753), done.


![Serve Positioning](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_08/serve_positioning.png)

![Serve Architecture](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_08/serve_arch.png)

![NLP API Architecture](https://raw.githubusercontent.com/maxpumperla/learning_ray/main/notebooks/images/chapter_08/nlp_api_arch.png)

In [3]:
from ray import serve

from transformers import pipeline


@serve.deployment
class SentimentAnalysis:
    def __init__(self):
        self._classifier = pipeline("sentiment-analysis")

    def __call__(self, request) -> str:
        input_text = request.query_params["input_text"]
        return self._classifier(input_text)[0]["label"]

In [4]:
basic_deployment = SentimentAnalysis.bind()

In [9]:
# Run this in a separate process to avoid any blocking:
! serve run --non-blocking app:basic_deployment

2025-12-13 15:43:42,270	INFO scripts.py:511 -- Running import path: 'app:basic_deployment'.
2025-12-13 15:43:46.228056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765640626.250480    5139 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765640626.256987    5139 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765640626.274372    5139 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765640626.274404    5139 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than o

In [7]:
import requests

print(requests.get(
    "http://localhost:8000/", params={"input_text": "Hello friend!"}
).json())

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /?input_text=Hello+friend%21 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e7543883770>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [8]:
from fastapi import FastAPI

app = FastAPI()


@serve.deployment
@serve.ingress(app)
class SentimentAnalysis:
    def __init__(self):
        self._classifier = pipeline("sentiment-analysis")

    @app.get("/")
    def classify(self, input_text: str) -> str:
        return self._classifier(input_text)[0]["label"]


fastapi_deployment = SentimentAnalysis.bind()

In [10]:
app = FastAPI()


@serve.deployment(num_replicas=2, ray_actor_options={"num_cpus": 2})
@serve.ingress(app)
class SentimentAnalysis:
    def __init__(self):
        self._classifier = pipeline("sentiment-analysis")

    @app.get("/")
    def classify(self, input_text: str) -> str:
        import os
        print("from process:", os.getpid())
        return self._classifier(input_text)[0]["label"]


scaled_deployment = SentimentAnalysis.bind()

In [11]:
! serve run --non-blocking app:scaled_deployment

2025-12-13 15:47:25,549	INFO scripts.py:511 -- Running import path: 'app:scaled_deployment'.
2025-12-13 15:47:29.532947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765640849.557124    6624 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765640849.564046    6624 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765640849.582076    6624 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765640849.582115    6624 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than 

In [12]:
app = FastAPI()


@serve.deployment
@serve.ingress(app)
class SentimentAnalysis:
    def __init__(self):
        self._classifier = pipeline("sentiment-analysis")

    @serve.batch(max_batch_size=10, batch_wait_timeout_s=0.1)
    async def classify_batched(self, batched_inputs):
        print("Got batch size:", len(batched_inputs))
        results = self._classifier(batched_inputs)
        return [result["label"] for result in results]

    @app.get("/")
    async def classify(self, input_text: str) -> str:
        return await self.classify_batched(input_text)


batched_deployment = SentimentAnalysis.bind()

In [17]:
import asyncio
import ray
from ray import serve
from app import batched_deployment

handle = serve.run(batched_deployment)
results = asyncio.gather(*[handle.classify.remote("sample text") for _ in range(10)])

INFO 2025-12-13 16:03:27,652 serve 667 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
(ServeController pid=9448) INFO 2025-12-13 16:03:27,688 controller 9448 -- Deploying new version of Deployment(name='SentimentAnalysis', app='default') (initial target replicas: 1).
(ServeController pid=9448) INFO 2025-12-13 16:03:27,809 controller 9448 -- Stopping 1 replicas of Deployment(name='SentimentAnalysis', app='default') with outdated versions.
(ServeController pid=9448) INFO 2025-12-13 16:03:27,809 controller 9448 -- Adding 1 replica to Deployment(name='SentimentAnalysis', app='default').
(ServeController pid=9448) INFO 2025-12-13 16:03:29,966 controller 9448 -- Replica(id='55ev2x96', deployment='SentimentAnalysis', app='default') is stopped.
(ServeReplica:default:SentimentAnalysis pid=12216) 2025-12-13 16:03:47.928790: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register facto

In [19]:
results

<_GatheringFuture finished result=['POSITIVE', 'POSITIVE', 'POSITIVE', 'POSITIVE', 'POSITIVE', 'POSITIVE', ...]>

In [20]:
@serve.deployment
class DownstreamModel:
    def __call__(self, inp: str):
        return "Hi from downstream model!"


@serve.deployment
class Driver:
    def __init__(self, downstream):
        self._d = downstream

    async def __call__(self, *args) -> str:
        return await self._d.remote()


downstream = DownstreamModel.bind()
driver = Driver.bind(downstream)

In [21]:
@serve.deployment
class DownstreamModel:
    def __init__(self, my_val: str):
        self._my_val = my_val

    def __call__(self, inp: str):
        return inp + "|" + self._my_val


@serve.deployment
class PipelineDriver:
    def __init__(self, model1, model2):
        self._m1 = model1
        self._m2 = model2

    async def __call__(self, *args) -> str:
        intermediate = self._m1.remote("input")
        final = self._m2.remote(intermediate)
        return await final


m1 = DownstreamModel.bind("val1")
m2 = DownstreamModel.bind("val2")
pipeline_driver = PipelineDriver.bind(m1, m2)

In [ ]:
@serve.deployment
class DownstreamModel:
    def __init__(self, my_val: str):
        self._my_val = my_val

    def __call__(self):
        return self._my_val


@serve.deployment
class BroadcastDriver:
    def __init__(self, model1, model2):
        self._m1 = model1
        self._m2 = model2

    async def __call__(self, *args) -> str:
        output1, output2 = self._m1.remote(), self._m2.remote()
        return [await output1, await output2]


m1 = DownstreamModel.bind("val1")
m2 = DownstreamModel.bind("val2")
broadcast_driver = BroadcastDriver.bind(m1, m2)

In [22]:
@serve.deployment
class DownstreamModel:
    def __init__(self, my_val: str):
        self._my_val = my_val

    def __call__(self):
        return self._my_val


@serve.deployment
class ConditionalDriver:
    def __init__(self, model1, model2):
        self._m1 = model1
        self._m2 = model2

    async def __call__(self, *args) -> str:
        import random
        if random.random() > 0.5:
            return await self._m1.remote()
        else:
            return await self._m2.remote()


m1 = DownstreamModel.bind("val1")
m2 = DownstreamModel.bind("val2")
conditional_driver = ConditionalDriver.bind(m1, m2)

In [2]:
from typing import Optional

import wikipedia


def fetch_wikipedia_page(search_term: str) -> Optional[str]:
    results = wikipedia.search(search_term)
    # If no results, return to caller.
    if len(results) == 0:
        return None

    # Get the page for the top result.
    return wikipedia.page(results[0]).content

In [3]:
from ray import serve
from transformers import pipeline
from typing import List


@serve.deployment
class SentimentAnalysis:
    def __init__(self):
        self._classifier = pipeline("sentiment-analysis")

    @serve.batch(max_batch_size=10, batch_wait_timeout_s=0.1)
    async def is_positive_batched(self, inputs: List[str]) -> List[bool]:
        results = self._classifier(inputs, truncation=True)
        return [result["label"] == "POSITIVE" for result in results]

    async def __call__(self, input_text: str) -> bool:
        return await self.is_positive_batched(input_text)

In [4]:
@serve.deployment(num_replicas=2)
class Summarizer:
    def __init__(self, max_length: Optional[int] = None):
        self._summarizer = pipeline("summarization")
        self._max_length = max_length

    def __call__(self, input_text: str) -> str:
        result = self._summarizer(
            input_text, max_length=self._max_length, truncation=True)
        return result[0]["summary_text"]

In [5]:
@serve.deployment
class EntityRecognition:
    def __init__(self, threshold: float = 0.90, max_entities: int = 10):
        self._entity_recognition = pipeline("ner")
        self._threshold = threshold
        self._max_entities = max_entities

    def __call__(self, input_text: str) -> List[str]:
        final_results = []
        for result in self._entity_recognition(input_text):
            if result["score"] > self._threshold:
                final_results.append(result["word"])
            if len(final_results) == self._max_entities:
                break

        return final_results

In [17]:
from pydantic import BaseModel
from typing import TypedDict


# class Response(BaseModel):
#     success: bool
#     message: str = ""
#     summary: str = ""
#     named_entities: list[str] = []

class Response(TypedDict, total=False):
    success: bool
    message: str
    summary: str
    named_entities: list[str]


In [23]:
from fastapi import FastAPI

app = FastAPI()


@serve.deployment
@serve.ingress(app)
class NLPPipelineDriver:
    def __init__(self, sentiment_analysis, summarizer, entity_recognition):
        self._sentiment_analysis = sentiment_analysis
        self._summarizer = summarizer
        self._entity_recognition = entity_recognition

    @app.get("/", response_model=Response)
    async def summarize_article(self, search_term: str) -> Response:
        # Fetch the top page content for the search term if found.
        page_content = fetch_wikipedia_page(search_term)
        if page_content is None:
            return Response(success=False, message="No pages found.")

        # Conditionally continue based on the sentiment analysis.
        is_positive = await self._sentiment_analysis.remote(page_content)
        if not is_positive:
            return Response(success=False, message="Only positivitiy allowed!")

        # Query the summarizer and named entity recognition models in parallel.
        summary_result = self._summarizer.remote(page_content)
        entities_result = self._entity_recognition.remote(page_content)
        return Response(
            success=True,
            summary=await summary_result,
            named_entities=await entities_result
        )

In [24]:
sentiment_analysis = SentimentAnalysis.bind()
summarizer = Summarizer.bind()
entity_recognition = EntityRecognition.bind(threshold=0.95, max_entities=5)
nlp_pipeline_driver = NLPPipelineDriver.bind(
    sentiment_analysis, summarizer, entity_recognition)

In [ ]:
import ray
from ray import serve

ray.init(
    ignore_reinit_error=True,
    include_dashboard=False,  # Disable dashboard
    _temp_dir="/tmp/ray"
)

serve.start()

handle = serve.run(nlp_pipeline_driver, route_prefix="/")

2025-12-13 16:41:41,769	INFO worker.py:1855 -- Calling ray.init() again after it has already been called.
INFO 2025-12-13 16:42:09,518 serve 18580 -- Started Serve in namespace "serve".
INFO 2025-12-13 16:42:09,618 serve 18580 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.


In [ ]:
import requests


print(requests.get(
    "http://localhost:8000/", params={"search_term": "rayserve"}
).text)

In [ ]:
print(requests.get(
    "http://localhost:8000/", params={"search_term": "war"}
).text)

In [ ]:
print(requests.get(
    "http://localhost:8000/", params={"search_term": "physicist"}
).text)